In [ ]:
import rioxarray as rxr
import xarray as xr


year_list = [2019, 2020, 2021, 2022, 2023, 2024]
image_info = ['Region2', 'Region2a']
# 2. 读取每个raster的第6个波段（波段索引从1开始）
band7_list = []
for year in year_list:
    FB_raster_path = f"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}/{image_info[1]}_{year}_FB6_new9.tif"
    img = rxr.open_rasterio(FB_raster_path)
    band7 = img.sel(band=7)     # 选取第6波段
    band7_list.append(band7)

# 3. 相加
sum_band6 = xr.concat(band7_list, dim="raster").sum(dim="raster")

# 4. 保存结果
sum_band6.rio.to_raster(f"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\sum_band6.tif")

print("✅ 已输出 sum_band6.tif")


✅ 已输出 sum_band6.tif


In [3]:
import rioxarray as rxr
import xarray as xr
import os

# 定义年份和区域信息
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
image_info = ['Region2', 'Region2a']

# 参考文件路径（用2019年的文件作为基准）
ref_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year_list[0]}\{image_info[1]}_{year_list[0]}_FB6_new9.tif"

# 打开参考影像
ref = rxr.open_rasterio(ref_path)
ref_band7 = ref.sel(band=7)

# 存放波段
band7_list = []

for year in year_list:
    FB_raster_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}\{image_info[1]}_{year}_FB6_new9.tif"
    
    if not os.path.exists(FB_raster_path):
        print(f"⚠️ 文件不存在: {FB_raster_path}")
        continue

    img = rxr.open_rasterio(FB_raster_path)
    band7 = img.sel(band=7)

    # ✅ 强制重投影和对齐到参考栅格
    band7 = band7.rio.reproject_match(ref_band7)

    band7_list.append(band7)

# 相加
sum_band7 = xr.concat(band7_list, dim="raster").sum(dim="raster")

# 继承坐标系属性
sum_band7 = sum_band7.rio.write_crs(ref_band7.rio.crs)

# 输出路径
output_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\sum_band7.tif"

# 保存
sum_band7.rio.to_raster(output_path)

print(f"✅ 已输出: {output_path}")


✅ 已输出: G:\ICESat-2\Region2_map\Region2a_map\sum_band7.tif


In [94]:
import rioxarray as rxr
import xarray as xr
import os
import numpy as np

# 定义年份和区域信息
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
image_info = ['Region5', 'Region5b']

# 参考文件路径（2019年的文件作为基准）
ref_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year_list[0]}\{image_info[1]}_{year_list[0]}_FB6_new9.tif"

# 打开参考影像
ref = rxr.open_rasterio(ref_path)
ref_band7 = ref.sel(band=7)

# 存放波段
band7_list = []

# 逐年读取并对齐
for year in year_list:
    FB_raster_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}\{image_info[1]}_{year}_FB6_new9.tif"

    if not os.path.exists(FB_raster_path):
        print(f"⚠️ 文件不存在: {FB_raster_path}")
        continue

    img = rxr.open_rasterio(FB_raster_path)
    band7 = img.sel(band=7)
    
    # ✅ 对齐投影/分辨率/范围
    band7 = band7.rio.reproject_match(ref_band7)
    band7_list.append(band7)

# 计算 sum_band7
sum_band7 = xr.concat(band7_list, dim="raster").sum(dim="raster")
sum_band7 = sum_band7.rio.write_crs(ref_band7.rio.crs)

# 保存总和结果
output_sum_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\sum_band7.tif"
sum_band7.rio.to_raster(output_sum_path)
print(f"✅ 已输出: {output_sum_path}")

# 生成掩膜： <3 → 0, ≥3 → 1
mask = xr.where(sum_band7 <= 0, 0, 1)
mask = mask.rio.write_crs(sum_band7.rio.crs)

# 输出掩膜
mask_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\sum_band7_mask.tif"
mask.rio.to_raster(mask_path)
print(f"✅ 掩膜已生成: {mask_path}")

# -----------------------------
#  应用掩膜到每个原始栅格
# -----------------------------Region2a_2019_4_RF_FB6_new9_bys_join.tif


for year in year_list:
    FB_raster_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}\{image_info[1]}_{year}_4_RF_FB6_new9_bys.tif"
    FB_raster_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}\{image_info[1]}_{year}_clipped.tif"
    if not os.path.exists(FB_raster_path):
        print(FB_raster_path, 'is not exit')
        continue

    img = rxr.open_rasterio(FB_raster_path)
    band7 = img.sel(band=1).rio.reproject_match(sum_band7)
    
    # 掩膜处理（只保留 mask==1 的像元）
    masked_band7 = band7.where(mask == 1)

    # 输出路径
    out_masked_path = fr"G:\ICESat-2\{image_info[0]}_map\{image_info[1]}_map\A{year}\{image_info[1]}_{year}_clipped2.tif"
    masked_band7.rio.to_raster(out_masked_path)
    
    print(f"✅ 掩膜已生成: {out_masked_path}")

✅ 已输出: G:\ICESat-2\Region5_map\Region5b_map\sum_band7.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\sum_band7_mask.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2019\Region5b_2019_clipped2.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2020\Region5b_2020_clipped2.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2021\Region5b_2021_clipped2.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2022\Region5b_2022_clipped2.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2023\Region5b_2023_clipped2.tif
✅ 掩膜已生成: G:\ICESat-2\Region5_map\Region5b_map\A2024\Region5b_2024_clipped2.tif


In [92]:
import rioxarray as rxr
import geopandas as gpd

# 1. 读取栅格
for year in [2023, 2024]:
    raster = rxr.open_rasterio(f"G:\ICESat-2\Region4_map\Severn_map\A{year}\Severn_{year}_4_RF_FB6_new9_bys.tif")

    # 2. 读取ROI矢量
    roi = gpd.read_file("G:\ICESat-2\Region4_map\Severn_map\A2022\Severn_clip.shp")

    # 3. 确保CRS一致
    if roi.crs != raster.rio.crs:
        roi = roi.to_crs(raster.rio.crs)

    # 4. 裁剪
    clipped = raster.rio.clip(roi.geometry, roi.crs)

    # 5. 保存结果
    clipped.rio.to_raster(f"G:\ICESat-2\Region4_map\Severn_map\A{year}/Severn_{year}_4_RF_FB6_new9_bys_C.tif")
    print("✅ 裁剪完成，结果已保存为 raster_clipped.tif")


✅ 裁剪完成，结果已保存为 raster_clipped.tif
✅ 裁剪完成，结果已保存为 raster_clipped.tif


In [106]:
import colorsys

# HSV 输入
h, s, v = 0.15, 0.05, 0.87  # H 0–1, S 0–1, V 0–1
r, g, b = colorsys.hsv_to_rgb(h, s, v)

# 转换为 0–255 整数
R, G, B = int(r*81), int(g*13), int(b*97)

# 转成 Hex
hex_color = "#{:02X}{:02X}{:02X}".format(R, G, B)
print(hex_color)


#460B50


In [ ]:
['#644222', '#AE762A', '#DDDC9D','#4F3691', '#460B50']

In [ ]:
['#75201F', '#D08009', '#DDDC9D', '#404595', '#211B64']